# Workshop: Batch & Stream Ingestion

Practice COPY INTO, Auto Loader, schema evolution, and streaming patterns with hands-on exercises.

| Duration | Format | Difficulty |
|---|---|---|
| 30 min | Hands-on Workshop | Intermediate |

**Prerequisites:** 02 — Batch & Stream Ingestion Demo

<!-- PART1-FUNDAMENTAL -->

---

# PART 1 — FUNDAMENTAL (Fundamental Level)

---

<!-- LAB-SCENARIO -->

## Scenario

> *"New files keep arriving in cloud storage (S3/ADLS). Instead of full re-scans, you need to implement Auto Loader (`cloudFiles`) to process only new files incrementally — with schema inference and checkpointing."*

<!-- LAB-OBJECTIVES -->

## Learning Objectives

After completing this lab you will be able to:

- Configure Auto Loader source using `cloudFiles` format options
- Enable schema inference and automatic schema evolution
- Use `trigger(availableNow=True)` for batch-style incremental runs
- Implement checkpointing for fault tolerance and restartability
- Monitor Auto Loader progress in Spark UI and query metrics

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Prepare landing zone and checkpoint paths
landing_path = f"{DATASET_PATH}/orders/stream"
checkpoint_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint"
schema_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema"
target_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_stream"

# Clean up from previous runs
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

# Ensure customers table exists (needed for Task 8: Stream-Static Join)
customers_path = f"{DATASET_PATH}/customers/customers.csv"
if not spark.catalog.tableExists(f"{CATALOG}.{BRONZE_SCHEMA}.customers"):
    spark.read.format("csv").option("header", True).option("inferSchema", True) \
        .load(customers_path).write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
    print(f"Created: {CATALOG}.{BRONZE_SCHEMA}.customers")

print(f"Landing path: {landing_path}")
print(f"Target table: {target_table}")
print(f"Files available: {[f.name for f in dbutils.fs.ls(landing_path)]}")

---
## Task 1: COPY INTO (Batch Ingestion)

Use `COPY INTO` to load JSON files from the landing zone into a Delta table.

**What you need to do:** Fill in the `FILEFORMAT` parameter.


**Guidance — Task 01**

The goal is to load raw JSON files into a Delta table using the simplest batch ingestion command — `COPY INTO`.

**How COPY INTO works**
`COPY INTO` is a SQL command that reads files from a source path and appends them to an existing Delta table. It tracks which files have been loaded using file-level metadata, so running it twice on the same files produces zero new rows. The command requires a `FILEFORMAT` parameter and optional `FORMAT_OPTIONS`.

**What the setup code does**
The `CREATE OR REPLACE TABLE` statement defines the target schema. You need to fill in the file format and the format options — use `mergeSchema = 'true'` to allow minor schema differences between files.

**Things to think about**
- Why does `COPY INTO` need the target table to exist before loading?
- How does it differ from a simple `INSERT INTO ... SELECT * FROM read_files(...)`?
- What happens if the source files contain a column not in the table schema?


In [ ]:
# First create the target table
spark.sql(f"""
    CREATE OR REPLACE TABLE {target_table}
    (order_id STRING, customer_id STRING, product_id STRING, store_id STRING,
     order_datetime STRING, quantity BIGINT, unit_price DOUBLE,
     discount_percent BIGINT, total_amount DOUBLE, payment_method STRING)
""")

# TODO: Use COPY INTO to load JSON files
spark.sql(f"""
    COPY INTO {target_table}
    FROM '{landing_path}'
    FILEFORMAT = _______
    FORMAT_OPTIONS ___________
""")

count_after_copy = spark.table(target_table).count()
print(f"Rows after COPY INTO: {count_after_copy}")

In [ ]:
# -- Validation --
assert count_after_copy > 0, "COPY INTO should have loaded data"
print(f"Task 1 OK: {count_after_copy} rows loaded via COPY INTO")

---
## Task 2: Verify COPY INTO Idempotency

Run COPY INTO again on the **same files**. Observe that 0 new rows are loaded — COPY INTO tracks which files were already processed.

**What you need to do:** Rewrite the same COPY INTO from Task 1 (no changes to the query) and verify the row count stays the same.


**Guidance — Task 02**

The goal is to prove that `COPY INTO` is **idempotent** — running it again on the same files loads zero new rows.

**Why idempotency matters**
In production, pipelines can fail mid-run and need to be safely retried. If the ingestion command is not idempotent, retrying would create duplicates. `COPY INTO` avoids this by storing the list of already-loaded file paths as metadata on the target table.

**What to observe**
Run the cell and compare the counts. The difference should be exactly zero. This is the key property that makes `COPY INTO` safe for scheduled batch jobs — even if the scheduler fires twice.

**Things to think about**
- Where does COPY INTO store the "already loaded" file list — in the Delta log or externally?
- If you `DROP` and recreate the table, will COPY INTO re-load the same files?

In [ ]:
# TODO: Re-run the same COPY INTO from Task 1 (no changes needed — just run it again)
# Use the same query you wrote in Task 1

count_after_rerun = spark.table(target_table).count()
print(f"Rows after re-run: {count_after_rerun} (was {count_after_copy})")
print(f"New rows loaded: {count_after_rerun - count_after_copy}")


In [ ]:
# -- Validation --
assert count_after_rerun == count_after_copy, "COPY INTO should be idempotent - no new rows!"
print("Task 2 OK: COPY INTO is idempotent. 0 new rows on re-run.")

---
## Task 3: Auto Loader — Configure Stream

Set up Auto Loader (`cloudFiles`) to read JSON files from the landing zone.

**What you need to do:** Configure all required `cloudFiles` options to enable schema inference and column type detection.


**Guidance — Task 03**

The goal is to configure Auto Loader — Databricks' recommended way to ingest files at scale using Structured Streaming.

**How Auto Loader works**
Auto Loader uses the `cloudFiles` format as its Structured Streaming source. The actual file format (JSON, CSV, Parquet) is specified separately via the `cloudFiles.format` option. This two-level approach allows Auto Loader to handle file discovery and tracking independently of file parsing.

**Schema management**
The `schemaLocation` option tells Auto Loader where to persist the inferred schema. On first run, it scans the files and writes the schema to this path. On subsequent runs, it reuses the cached schema — avoiding expensive re-inference. The `inferColumnTypes` option promotes strings to proper types (INT, DOUBLE, etc.).

**Things to think about**
- Why does Auto Loader need a separate `schemaLocation` path?
- What happens if you delete the schema location and re-run?
- How does `cloudFiles` compare to `COPY INTO` for millions of files?

In [ ]:
# Reset target for Auto Loader test
al_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_autoloader"
spark.sql(f"DROP TABLE IF EXISTS {al_target}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

In [ ]:
# TODO: Configure Auto Loader readStream
df_stream = (
    spark.readStream
    # TODO: set the correct format for Auto Loader
    # TODO: set the file format option
    # TODO: set schema location
    # TODO: enable column type inference
    .load(landing_path)
)


In [ ]:
# -- Validation --
assert df_stream.isStreaming, "Should be a streaming DataFrame"
print(f"Task 3 OK: Streaming DataFrame configured with schema: {df_stream.schema.fieldNames()}")

---
## Task 4: Write Stream with trigger(availableNow=True)

Write the streaming DataFrame to a Delta table using a trigger that processes all available files and stops.

**What you need to do:** Complete the `writeStream` chain — all parameters must be set correctly.


**Guidance — Task 04**

The goal is to write the streaming DataFrame to a Delta table using the `availableNow` trigger mode.

**Trigger modes explained**
Structured Streaming supports several trigger modes. `availableNow=True` is the most common for batch-like pipelines — it processes all pending files in multiple micro-batches and then stops. This is different from `once=True` (legacy, single micro-batch) and `processingTime` (continuous, runs forever at intervals).

**The writeStream chain**
The writer follows a builder pattern: `.format("delta")` → `.outputMode("append")` → `.option("checkpointLocation", ...)` → `.trigger(...)` → `.toTable(...)`. The checkpoint is essential — it stores which files and offsets have been processed. Without it, every run would reprocess everything.

**Things to think about**
- When would you use `processingTime="10 seconds"` instead of `availableNow=True`?
- What is stored in the checkpoint directory — and what happens if you delete it?

In [ ]:
# TODO: Write stream to Delta table
# Required: format, outputMode, checkpointLocation, trigger (process all available and stop), target table
query = (
    df_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    # TODO: set trigger to process all available files and stop
    .toTable(al_target)
)

query.awaitTermination()
print(f"Stream completed. Rows loaded: {spark.table(al_target).count()}")


In [ ]:
# -- Validation --
al_count = spark.table(al_target).count()
assert al_count > 0, "Auto Loader should have loaded data"
print(f"Task 4 OK: {al_count} rows loaded via Auto Loader")

---
## Task 5: Incremental Processing

Re-run the stream with the **same checkpoint**. Since no new files arrived, 0 new rows should be processed — this proves checkpoint-based tracking works.

**What you need to do:** Repeat Tasks 3 and 4 — configure the readStream and writeStream using the **same checkpoint path** (`checkpoint_path`). Store the query in `query2` and await termination.


**Guidance — Task 05**

The goal is to verify that checkpoint-based tracking works — re-running the stream with the same checkpoint processes zero new rows.

**How checkpoints enable incremental processing**
When Auto Loader runs with a checkpoint, it records the offsets (which files it has seen) and the committed micro-batch IDs. On the next run, it only looks for files newer than what was previously recorded. This is what makes streaming pipelines truly incremental — even across cluster restarts.

**Things to think about**
- What is the difference between COPY INTO's file tracking and Auto Loader's checkpoint tracking?
- In production, where should you store checkpoints — local disk, DBFS, or cloud storage?

In [ ]:
# TODO: Re-run the stream using the same readStream configuration as in Task 3
# and the same writeStream configuration as in Task 4.
# Use the SAME checkpoint path — this is what makes it incremental.
# Store the new query result in query2 and await termination.

al_count2 = spark.table(al_target).count()
print(f"Rows after re-run: {al_count2} (was {al_count})")
print(f"New rows: {al_count2 - al_count}")


In [ ]:
# -- Validation --
assert al_count2 == al_count, f"Should be 0 new rows, but got {al_count2 - al_count}"
print("Task 5 OK: Incremental processing verified. 0 new rows on re-run (checkpoint works!)")

---
## Task 6: Metadata Columns

Add metadata columns to streaming data for **debugging and auditing** in production.

**What you need to do:** Enrich the stream with `_processing_time` (processing timestamp) and `_source_file` (source file path) columns before writing to Delta.


**Guidance — Task 06**

The goal is to enrich streaming data with **metadata columns** that are essential for debugging and auditing in production.

**Why metadata matters**
In a multi-source pipeline, you often need to know *when* a row was processed and *which file* it came from. `current_timestamp()` captures the processing time, and `_metadata.file_path` (a hidden Spark column available in file-based sources) gives the source file path. These columns make troubleshooting much easier when data issues arise.

**The `_metadata` struct**
Spark automatically provides a `_metadata` struct column for file-based data sources. It contains fields like `file_path`, `file_name`, `file_size`, and `file_modification_time`. You access them via `col("_metadata.file_path")`.

**Things to think about**
- Why use `current_timestamp()` instead of `_metadata.file_modification_time`?
- Would `_metadata` work the same way when reading from a Delta table (not files)?

In [ ]:
from pyspark.sql.functions import current_timestamp, col

# Paths and reset
metadata_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata"
metadata_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_metadata"
metadata_schema = f"/tmp/{CATALOG}/lab05/schema_metadata"

spark.sql(f"DROP TABLE IF EXISTS {metadata_target}")
dbutils.fs.rm(metadata_checkpoint, True)
dbutils.fs.rm(metadata_schema, True)

In [ ]:
# TODO: Configure Auto Loader with metadata columns
df_with_metadata = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", metadata_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    # TODO: add _processing_time column (processing timestamp)
    # TODO: add _source_file column (path of the source file)
)


In [ ]:
# Write enriched stream to Delta
(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", metadata_checkpoint)
    .trigger(availableNow=True)
    .toTable(metadata_target)
    .awaitTermination()
)

print(f"Written to: {metadata_target}")
display(spark.table(metadata_target).limit(5))

In [ ]:
# -- Validation --
meta_cols = spark.table(metadata_target).columns
assert "_processing_time" in meta_cols, "Missing '_processing_time' — did you use current_timestamp()?"
assert "_source_file" in meta_cols, "Missing '_source_file' — did you use _metadata.file_path?"
print(f"Task 6 OK: Metadata columns added — {meta_cols}")

---
## Task 7: Schema Evolution — Rescued Data

Configure Auto Loader with a **partial schema** (only 2 columns). Extra columns from the source files must not be lost.

**What you need to do:**
1. Define a partial schema with only `order_id` and `customer_id`
2. Configure the appropriate `schemaEvolutionMode` so that extra columns are captured rather than dropped
3. Verify that `_rescued_data` column is present and populated


**Guidance — Task 07**

The goal is to demonstrate Auto Loader's **rescue mode** — handling schema mismatches without losing data.

**The schema evolution problem**
In production, source files evolve — new columns appear, types change. If you define a partial schema (only 2 columns), what happens to the other 8+ columns? With `schemaEvolutionMode = "rescue"`, Auto Loader captures them in a special `_rescued_data` column as a JSON string. No data is lost.

**Three schema evolution modes**
- `addNewColumns` — merges new columns into the schema automatically
- `rescue` — extra columns go into `_rescued_data`, known columns parsed normally
- `failOnNewColumns` — stream fails if unknown columns appear (strictest)

**Using `.schema()` with Auto Loader**
When you pass a partial schema via `.schema(partial_schema)`, Auto Loader applies it *instead of* inferring. Combined with rescue mode, this gives you a stable schema while preserving unexpected data.

**Things to think about**
- When would you prefer `addNewColumns` over `rescue`?
- How would you extract specific fields from the `_rescued_data` JSON string?

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

rescue_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued"
rescue_checkpoint = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint_rescue"
rescue_schema = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema_rescue"

spark.sql(f"DROP TABLE IF EXISTS {rescue_target}")
dbutils.fs.rm(rescue_checkpoint, True)
dbutils.fs.rm(rescue_schema, True)

# TODO: Define a partial schema with only order_id and customer_id (both StringType)
partial_schema = ________


In [ ]:
# TODO: Configure Auto Loader with the appropriate schema evolution mode
# and apply the partial schema
df_rescued = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", rescue_schema)
    # TODO: set schema evolution mode so extra columns are captured in _rescued_data
    # TODO: apply partial_schema
    .load(landing_path)
)


In [ ]:
# Write with rescued data
(
    df_rescued
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", rescue_checkpoint)
    .trigger(availableNow=True)
    .toTable(rescue_target)
    .awaitTermination()
)

print(f"Written to: {rescue_target}")
display(spark.table(rescue_target).limit(5))

In [ ]:
# -- Validation --
rescue_cols = spark.table(rescue_target).columns
assert "_rescued_data" in rescue_cols, "Missing '_rescued_data' column — did you set schemaEvolutionMode to 'rescue'?"
rescue_count = spark.table(rescue_target).filter("_rescued_data IS NOT NULL").count()
assert rescue_count > 0, "Expected rescued data for columns not in partial schema"
print(f"Task 7 OK: {rescue_count} rows with rescued data (extra columns captured in _rescued_data)")

---
## Task 8: Stream-Static Join

Join **streaming orders** with a **static customers** table to enrich the stream. The static side is re-read on every micro-batch.

**What you need to do:** Implement a stream-static left join on `customer_id` and write the result to `join_target`.


**Guidance — Task 08**

The goal is to **join streaming data with a static table** — one of the most common patterns in production pipelines.

**Stream-static join pattern**
In a stream-static join, one side is a streaming DataFrame (orders) and the other is a regular DataFrame (customers). Spark re-reads the static side on every micro-batch, so if the customer table gets updated between batches, the stream picks up the latest values automatically.

**Join mechanics**
Use `.join(static_df, on="column_name", how="left")`. A `left` join ensures all orders are preserved even if a customer is missing. The result is a streaming DataFrame that you write to Delta as usual.

**Things to think about**
- Can you join two streaming DataFrames together? What restrictions apply?
- Why is `left` join safer than `inner` join for this use case?
- What would happen if the customers table had duplicate `customer_id` values?

In [ ]:
join_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched"
join_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_enriched"

spark.sql(f"DROP TABLE IF EXISTS {join_target}")
dbutils.fs.rm(join_checkpoint, True)

# Static side: read customers table
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

In [ ]:
# TODO: Read orders as a stream from target_table (Delta format)
orders_stream = (
    spark.readStream
    .format("________")
    .table(________)
)

# TODO: Join with customers_df on customer_id using a left join
enriched_stream = orders_stream.join(
    ________,
    on="________",
    how="________"
)


In [ ]:
# TODO: Write the joined stream to join_target
(
    enriched_stream
    .writeStream
    .format("delta")
    .outputMode("________")
    .option("checkpointLocation", join_checkpoint)
    .trigger(availableNow=________)
    .toTable("________")
    .awaitTermination()
)

print(f"Stream-static join written to: {join_target}")
display(spark.table(join_target).limit(5))


In [ ]:
# -- Validation --
enriched_count = spark.table(join_target).count()
enriched_cols = spark.table(join_target).columns
assert enriched_count > 0, "No rows in enriched table"
assert "customer_id" in enriched_cols, "Missing customer_id"
assert any(c for c in enriched_cols if c not in spark.table(target_table).columns), \
    "Join didn't add any new columns — check join expression"
print(f"Task 8 OK: {enriched_count} enriched rows. Columns: {enriched_cols}")

<!-- PART2-ADVANCED -->

---

# PART 2 — ADVANCED (Bonus — If Time Permits)

> **For whom:** Participants with 1+ year of Spark/Databricks experience
>
> **Rules:**
> - No scaffold `= None` — write from scratch
> - Tasks described as JIRA tickets (requirements + acceptance criteria)
> - Complete **at least 2 of 4 challenges**
> - Time per challenge: 8-15 minutes

---

### Bug Hunt — Auto Loader Does Not Process New Files

After cluster restart, Auto Loader starts processing **from the beginning** (duplicates!).

```python
df_stream = (spark.readStream
             .format('cloudFiles')
             .option('cloudFiles.format', 'json')
             .option('cloudFiles.schemaLocation', SCHEMA_PATH)
             .load(LANDING_PATH))

query = (df_stream.writeStream
         .format('delta')
         .outputMode('append')
         .trigger(availableNow=True)
         .start(BRONZE_TABLE_PATH))  # bug: missing checkpointLocation
```

Find the bug, fix it, explain what the checkpoint stores.

**Acceptance criteria:**
- Identified bug with explanation
- Fixed code with `checkpointLocation`
- Explanation: what checkpoint stores (offset, schema)
- Demonstration: run 2x — second time 0 new rows

In [ ]:
# YOUR SOLUTION — Auto Loader Does Not Process New Files
# ------------------------------------------------------------



### Design From Scratch — Multi-Format Auto Loader

The `landing/` folder receives files from 3 systems simultaneously:
- `customers_*.csv` (separator `;`, header=true)
- `orders_*.json`
- `products_*.parquet`

Design a pipeline: one stream or multiple? Justify your choice.
Implement your chosen architecture.

**Acceptance criteria:**
- Justified decision: 1 stream with `pathGlobFilter` or 3 separate streams
- Each format goes to a separate Bronze table
- Each stream has its own `checkpointLocation`
- Column `_metadata.file_path` added to each table

In [ ]:
# YOUR SOLUTION — Multi-Format Auto Loader
# ------------------------------------------------------------



### Performance — availableNow vs processingTime — When to Use What?

You have 2 pipelines:
- **Pipeline A**: critical — new data must be visible within 30 seconds
- **Pipeline B**: batch overnight — processes 10 GB total per night

Choose the trigger mode for each, measure actual behavior on test data.

**Acceptance criteria:**
- Correct trigger for Pipeline A with justification
- Correct trigger for Pipeline B with justification
- Demonstrate `query.lastProgress['numInputRows']` and `'processedRowsPerSecond'`
- Written answer: when NOT to use `continuous` trigger

In [ ]:
# YOUR SOLUTION — availableNow vs processingTime — When to Use What?
# ------------------------------------------------------------



---

## Part 2 Summary

You have completed the Advanced section for **Batch & Stream Ingestion**.

**Reflection (optional):**
- Which challenge was the hardest and why?
- What would you change in your implementation?
- What approach would you use in production?

---

---
## Cleanup

In [ ]:
# Stop any active streams
for s in spark.streams.active:
    s.stop()

# Drop lab tables
for t in [target_table, al_target, 
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Clean temp paths
for p in [checkpoint_path, schema_path,
          f"/tmp/{CATALOG}/lab05/checkpoint_metadata",
          f"/tmp/{CATALOG}/lab05/schema_metadata",
          f"/tmp/{CATALOG}/lab05/checkpoint_rescue",
          f"/tmp/{CATALOG}/lab05/schema_rescue",
          f"/tmp/{CATALOG}/lab05/checkpoint_enriched"]:
    dbutils.fs.rm(p, True)

print("Lab cleanup complete")

---
## Lab Complete!

You have:
- Used COPY INTO for idempotent batch loading
- Configured Auto Loader (cloudFiles) for streaming ingestion
- Used trigger(availableNow=True) for incremental processing
- Verified checkpoint-based exactly-once guarantees
- Added metadata columns (`_processing_time`, `_source_file`) to streaming writes
- Used rescued data column for schema evolution handling
- Performed a stream-static join to enrich streaming data

| Feature | COPY INTO | Auto Loader |
|---------|-----------|-------------|
| Format | SQL command | readStream/writeStream |
| Scalability | Thousands of files | Millions of files |
| Schema evolution | Manual | Automatic (rescue column) |
| File tracking | SQL state | Checkpoint directory |
| Use case | Simple batch | File-based streaming |

> **Pro Tip:** Auto Loader uses `cloudFiles` format. COPY INTO is simpler but Auto Loader scales better (file notification mode for millions of files).

> **Next:** [02 -- Lakeflow Pipelines Workshop](02_lakeflow_pipelines_workshop.ipynb)

---

← [02 — Batch & Stream Ingestion](../Demo/02_batch_stream_ingestion_demo.ipynb) | **[ README](../../../README.md)** | [03 — Lakeflow Pipelines](../Demo/03_lakeflow_pipelines_demo.ipynb) →
